In [ ]:
"""Post-process the hourly AirNow-format AWS NetCDFs (Thailand + Philippines):

  1. Physical QC masks
       - relative_humidity outside [0, 100]            -> NaN
       - pressure below a plausibility floor           -> NaN
         (surface_pressure < 850 hPa; barometric_pressure < 637.5 mmHg = 850 hPa)
       - dewpoint > temperature                        -> NaN

  2. convert to consistent units 
  
  3. Derive humidity with MetPy where it's missing
       - dewpoint           from (temperature, relative_humidity)
       - relative_humidity  from (temperature, dewpoint)
     Existing measured values are kept; only NaN gaps are filled (and a missing
     variable is added outright, e.g. dewpoint for Thailand).

Requires MetPy:  conda install -c conda-forge metpy   (or pip install metpy)
"""


In [ ]:
# checks to run after generating NETCDFs

import numpy as np
import xarray as xr
from metpy.calc import (
    dewpoint_from_relative_humidity,
    relative_humidity_from_dewpoint,
)
from metpy.units import units


In [ ]:
FILES = [
    "/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/asiaaq_cs_06082026/preprocessing/thai_data/thai_aws_airnow.nc",
    "/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/asiaaq_cs_06082026/preprocessing/philippines_data/philippines_aws_airnow.nc",
]

PRESSURE_FLOOR_HPA = 800.0
PRESSURE_CEIL_HPA = 1100.0

CONC_RANGE = {
    "PM2.5": (0.0, 1000.0),   # ug/m3
    "PM10":  (0.0, 2000.0),   # ug/m3
    "O3":    (0.0, 300.0),    # ppbv
}

# consistent units 
VAR_UNITS = {
    "temperature":       "degC",
    "dewpoint":          "degC",
    "relative_humidity": "%",
    "surface_pressure":  "hPa",
    "wind_speed":        "m/s",
    "wind_direction":    "deg",
    "solar_radiation":   "W/m^2",
    "precip_accum":      "mm",
    "CO":                "ppmv",
    "NO2":               "ppbv",
    "O3":                "ppbv",
    "PM10":              "ug/m3",
    "PM2.5":             "ug/m3",
}

def normalize_units(ds):
    """
    thailand pressure in mmHg"""
    
    log = {}
    if "barometric_pressure" in ds:
        sp = ds["barometric_pressure"] #* MMHG_TO_HPA
        ds = ds.drop_vars("barometric_pressure")
        if "surface_pressure" in ds:        # don't clobber an existing hPa field
            ds["surface_pressure"] = ds["surface_pressure"].fillna(sp)
        else:
            ds["surface_pressure"] = sp
        log["barometric_pressure(mmHg) -> surface_pressure(hPa)"] = True
    return ds, log
    
def set_units(ds):
    
    for v, u in VAR_UNITS.items():
        if v in ds:
            ds[v].attrs["units"] = u
    return ds

def _n(da):
    return int(da.notnull().sum())

def apply_qc(ds):
    """Mask physically implausible values"""
    log = {}

    if "temperature" in ds:
        b = ds["temperature"]
        ds["temperature"] = b.where((b >= -20) & (b <= 60))   
        log["temperature outside [-20,60]C"] = _n(b) - _n(ds["temperature"])
            
    if "relative_humidity" in ds:
        b = ds["relative_humidity"]
        # keep (0, 100]; exclude 0 so MetPy dewpoint(T, RH=0) can't give -inf
        ds["relative_humidity"] = b.where((b > 0) & (b <= 100))
        log["RH outside (0,100]"] = _n(b) - _n(ds["relative_humidity"])

    if "wind_speed" in ds:
        b = ds["wind_speed"]
        ds["wind_speed"] = b.where(b >= 0)
        log["wind_speed < 0"] = _n(b) - _n(ds["wind_speed"])

    if "surface_pressure" in ds:
        b = ds["surface_pressure"]
    
        ds["surface_pressure"] = b.where(
            (b >= PRESSURE_FLOOR_HPA) & (b <= PRESSURE_CEIL_HPA)
        )
    
        log[
            f"surface_pressure outside [{PRESSURE_FLOOR_HPA}, {PRESSURE_CEIL_HPA}] hPa"
        ] = _n(b) - _n(ds["surface_pressure"])

    if "dewpoint" in ds:
        b = ds["dewpoint"]
        ds["dewpoint"] = b.where((b >= -30) & (b <= 40))
        log["dewpoint outside [-30,40]C"] = _n(b) - _n(ds["dewpoint"])

    # Concentration plausibility (PM2.5, PM10, O3): mask outside [lo, hi].
    for v, (lo, hi) in CONC_RANGE.items():
        if v in ds:
            b = ds[v]
            ds[v] = b.where((b >= lo) & (b <= hi))
            log[f"{v} outside [{lo},{hi}]"] = _n(b) - _n(ds[v])

    return ds, log

def _calc(func, *arrays, out_unit):
    """Run a MetPy func element-wise on the finite points only (NaN elsewhere).
    """
    base = arrays[0]
    out = np.full(base.shape, np.nan, dtype="float64")
    m = np.ones(base.shape, dtype=bool)
    for a in arrays:
        m &= np.isfinite(a)
    if m.any():
        q = func(*[a[m] for a in arrays])
        out[m] = np.asarray(q.to(out_unit).magnitude, dtype="float64")
    return out

def add_humidity(ds):
    """Add/fill dewpoint and relative_humidity via MetPy"""
    log = {}
    if "temperature" not in ds:
        return ds, log
    dims = ds["temperature"].dims
    coords = ds["temperature"].coords
    T = ds["temperature"].values

    if "relative_humidity" in ds:
        td = _calc(
            lambda t, rh: dewpoint_from_relative_humidity(
                t * units.degC, rh * units.percent),
            T, ds["relative_humidity"].values, out_unit="degC",
        )
        # Compute for EVERY time (T, RH) both exist -
        
        ds["dewpoint_qc"] = xr.DataArray(td, dims=dims, coords=coords)
        ds["dewpoint_qc"].attrs.update(
            units="degC",
            description="dewpoint computed from (T, RH) via MetPy wherever both exist",
        )
        log["dewpoint_qc computed"] = _n(ds["dewpoint_qc"])

    dew_src = "dewpoint" if "dewpoint" in ds else (
        "dewpoint_qc" if "dewpoint_qc" in ds else None)
    
    if dew_src is not None:
        rh = _calc(
            lambda t, d: relative_humidity_from_dewpoint(
                t * units.degC, d * units.degC),
            T, ds[dew_src].values, out_unit="percent",
        )
        ds["relative_humidity_qc"] = xr.DataArray(rh, dims=dims, coords=coords)
        ds["relative_humidity_qc"].attrs.update(
            units="%",
            description=f"RH computed from (T, {dew_src}) via MetPy wherever both exist",
        )
        log["relative_humidity_qc computed"] = _n(ds["relative_humidity_qc"])

    return ds, log
    

In [ ]:
for f in FILES:
    print("=" * 80)
    print(f)
    ds = xr.open_dataset(f).load()

    ds, unit_log = normalize_units(ds)   # mmHg to hPa, unify names

    # Snapshot valid-point counts so we can report how many points each
    # variable lost to QC (NaNs added by the masks).
    
    before = {v: _n(ds[v]) for v in ds.data_vars}
    ds, qc_log = apply_qc(ds)            # now pressure floor is uniform hPa
    qc_by_var = {v: before[v] - _n(ds[v]) for v in before
                 if before[v] - _n(ds[v]) > 0}

    ds, rh_log = add_humidity(ds)

    # Derived dewpoint (from RH<=100) is already <=T, but re-assert after
    # filling in case a measured value slipped through.
    if "dewpoint" in ds and "temperature" in ds:
        ds["dewpoint"] = ds["dewpoint"].where(ds["dewpoint"] <= ds["temperature"])

    # Rename the measured dewpoint so it doesn't collide with the model's
    # 'dewpoint' variable during MM pairing.
    if "dewpoint" in ds:
        ds = ds.rename({"dewpoint": "dewpoint_meas"})

    ds = set_units(ds)                   # stamp canonical units attrs

    print("  units       :", unit_log)
    print("  QC by rule  :", qc_log)
    print("  QC by var   :", qc_by_var, "| total:", sum(qc_by_var.values()))
    print("  humidity    :", rh_log)

    out = f[:-3] + "_qc.nc" if f.endswith(".nc") else f + "_qc.nc"
    ds.to_netcdf(out)
    print("  wrote       :", out)